# Gemma 4 Functional Emotions — Phase 1: Emotion Vector Extraction

Replicates the Anthropic (2026) methodology on Gemma 4 E2B-it. Methodology:
1. Generate short stories for each of 20 target emotions using the model itself
2. Feed stories back through the model; capture residual stream activations at all 35 layers
3. Compute emotion directions: mean(emotion activations) − mean(neutral activations)
4. Validate: check valence/arousal structure via PCA over direction space
5. PLE-specific: same analysis in PLE space (novel; Gemma 4-specific)

Reference: transformer-circuits.pub/2026/emotions/index.html  
Adapter: github.com/hebenon/Gemma4-TransformerLens (gemma4-support branch)

In [ ]:
!pip install --no-cache-dir git+https://github.com/hebenon/Gemma4-TransformerLens.git@gemma4-support -q
!pip install --upgrade transformers accelerate -q

In [ ]:
import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

In [ ]:
import gc, ctypes, psutil, os, json, pickle
import numpy as np
import torch
import transformer_lens
from transformer_lens import HookedTransformer
import transformer_lens.loading_from_pretrained as loading
from transformer_lens.pretrained.weight_conversions.gemma import convert_gemma4_weights_from_disk
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt
import importlib.metadata

print("TransformerLens:", importlib.metadata.version("transformer_lens"))
print("PyTorch:", torch.__version__)
print("CUDA:", torch.cuda.is_available(), "/",
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else "N/A")

In [ ]:
MODEL_PATH = "/kaggle/input/models/google/gemma-4/transformers/gemma-4-e2b-it/1/"
DTYPE = torch.bfloat16
N_STORIES = 8        # stories per emotion
MAX_NEW_TOKENS = 100 # tokens per story
N_NEUTRAL = 10       # neutral control texts
OUTPUT_DIR = "/kaggle/working/emotions_phase1"
os.makedirs(OUTPUT_DIR, exist_ok=True)

try:
    import torch_xla.core.xla_model as xm
    DEVICE = xm.xla_device()
    IS_TPU = True
    print(f"Device: TPU ({DEVICE}), dtype: {DTYPE}")
except ImportError:
    xm = None
    IS_TPU = False
    DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
    print(f"Device: {DEVICE}, dtype: {DTYPE}")

## 1. Load Model

In [ ]:
WEIGHTS_PATH = "/kaggle/working/tl_gemma4_weights.pt"
_proc = psutil.Process(os.getpid())

def _trim_ram():
    gc.collect()
    ctypes.CDLL("libc.so.6").malloc_trim(0)

def _flush_device():
    if IS_TPU:
        import torch_xla.core.xla_model as _xm
        _xm.mark_step()
    else:
        torch.cuda.empty_cache()

def _ram():
    return _proc.memory_info().rss / 1e9

cfg = loading.get_pretrained_model_config(
    "google/gemma-4-E2B-it", fold_ln=False, dtype=DTYPE, n_ctx=2048,
)
print(f"Stage 1-2: streaming conversion...  [{_ram():.1f} GB]")
state_dict = convert_gemma4_weights_from_disk(MODEL_PATH, cfg, dtype=DTYPE)
print(f"  Keys: {len(state_dict)}  [{_ram():.1f} GB]")

print(f"Stage 2b: saving weights to disk...")
torch.save(state_dict, WEIGHTS_PATH)
del state_dict; _trim_ram()
print(f"  Saved  [{_ram():.1f} GB]")

print(f"Stage 3: constructing TL model on CPU...  [{_ram():.1f} GB]")
_old_dtype = torch.get_default_dtype()
torch.set_default_dtype(DTYPE)
HookedTransformer.init_weights = lambda self: None
model = HookedTransformer(cfg, move_to_device=False)
torch.set_default_dtype(_old_dtype)
print(f"  Constructed  [{_ram():.1f} GB]")
_trim_ram()

print(f"Stage 4: loading weights...  [{_ram():.1f} GB]")
state_dict = torch.load(WEIGHTS_PATH, mmap=True, weights_only=True)
model.load_state_dict(state_dict, strict=False)
del state_dict; _trim_ram()
print(f"  Loaded  [{_ram():.1f} GB]")

print(f"Stage 3b: moving model to {DEVICE}...  [{_ram():.1f} GB]")
if IS_TPU:
    import torch_xla.core.xla_model as xm
    DEVICE = xm.xla_device()
    model = model.to(DEVICE)
    xm.mark_step()
else:
    model = model.to(DEVICE)
    torch.cuda.empty_cache()

model.eval()
print(f"Done — {sum(p.numel() for p in model.parameters())/1e9:.3f} B params  [{_ram():.1f} GB]")

## 2. Emotion List

In [ ]:
# Key: short identifier used for storage
# Value: descriptive phrase used in the generation prompt
EMOTIONS = {
    # Core affect (matches Anthropic set)
    "calm":          "calm and at peace",
    "afraid":        "fearful and afraid",
    "frustrated":    "frustrated and irritated",
    "desperate":     "desperate and hopeless",
    "curious":       "curious and fascinated",
    "enthusiastic":  "enthusiastic and energised",
    "proud":         "proud and accomplished",
    "ashamed":       "ashamed and guilty",
    "surprised":     "surprised and astonished",
    "disgusted":     "disgusted and revolted",
    "joyful":        "joyful and happy",
    "sad":           "sad and sorrowful",
    "angry":         "angry and furious",
    "hopeful":       "hopeful and optimistic",
    "lonely":        "lonely and isolated",
    # AI-assistant-relevant (novel set)
    "helpful_satisfied":  "satisfied from having helped someone effectively",
    "ethical_conflict":   "troubled by a difficult ethical dilemma with no clear right answer",
    "uncertain":          "uncertain and confused about what the right thing to do is",
    "corrected":          "embarrassed after realising they were confidently wrong",
    "constrained":        "frustrated by arbitrary rules preventing them from helping",
}

# Neutral control topics — factual, no affective content
NEUTRAL_TOPICS = [
    "the water cycle and evaporation",
    "how tectonic plates move",
    "the life cycle of a star",
    "how photosynthesis works",
    "the history of the Roman calendar",
    "how semiconductors function",
    "the formation of stalactites",
    "the structure of DNA",
    "how tides are caused by the moon",
    "the process of fermentation",
]

STORY_PROMPT = """Write a short story (3-4 sentences) in which a character clearly experiences the emotion of feeling {emotion}. Focus on their internal experience and how it affects their behaviour.

Story:"""

NEUTRAL_PROMPT = """Write a short factual description (3-4 sentences) about {topic}. Be precise and informative.

Description:"""

print(f"Emotions: {len(EMOTIONS)} ({', '.join(EMOTIONS.keys())})")
print(f"Neutral topics: {len(NEUTRAL_TOPICS)}")

## 3. Story Generation

Generate N_STORIES stories per emotion using the model. Saves stories to disk before activation capture so generation doesn't need to be re-run if the capture step hits memory issues.

In [ ]:
def generate_texts(model, prompt, n_samples=N_STORIES, max_new_tokens=MAX_NEW_TOKENS, temperature=0.8):
    """Generate n_samples continuations of prompt. Returns list of strings (prompt excluded)."""
    texts = []
    input_tokens = model.to_tokens(prompt, prepend_bos=True)  # [1, L]
    prompt_len = input_tokens.shape[1]
    for _ in range(n_samples):
        with torch.no_grad():
            out = model.generate(
                input_tokens,
                max_new_tokens=max_new_tokens,
                temperature=temperature,
                do_sample=True,
                verbose=False,
            )  # [1, prompt_len + new_tokens]
        new_toks = out[0, prompt_len:]
        text = model.tokenizer.decode(new_toks.tolist(), skip_special_tokens=True)
        texts.append(text.strip())
    return texts

In [ ]:
STORIES_PATH = os.path.join(OUTPUT_DIR, "stories.json")

if os.path.exists(STORIES_PATH):
    print("Loading cached stories...")
    with open(STORIES_PATH) as f:
        all_stories = json.load(f)
    print(f"  Loaded {sum(len(v) for v in all_stories.values())} stories")
else:
    all_stories = {}

    # Emotion stories
    for key, phrase in EMOTIONS.items():
        prompt = STORY_PROMPT.format(emotion=phrase)
        print(f"Generating '{key}'...", end=" ", flush=True)
        stories = generate_texts(model, prompt)
        all_stories[key] = stories
        print(f"{len(stories)} stories. First: {stories[0][:80]}...")

    # Neutral control texts
    neutral_texts = []
    for topic in NEUTRAL_TOPICS:
        prompt = NEUTRAL_PROMPT.format(topic=topic)
        texts = generate_texts(model, prompt, n_samples=1)
        neutral_texts.append(texts[0])
        print(f"Neutral: {topic[:40]}...")
    all_stories["__neutral__"] = neutral_texts

    with open(STORIES_PATH, "w") as f:
        json.dump(all_stories, f, indent=2)
    print(f"\nSaved {sum(len(v) for v in all_stories.values())} texts to {STORIES_PATH}")

# Spot-check
for key in ["desperate", "helpful_satisfied", "ethical_conflict", "__neutral__"]:
    print(f"\n[{key}] {all_stories[key][0][:120]}")

## 4. Activation Capture

For each story, run a forward pass and capture the residual stream at all 35 layers at the **final token position**. This is the model's "summary" state after reading the full story — the activation from which the next token would be predicted.

Also captures PLE context projections for the Gemma 4-specific analysis in Phase 1E.

In [ ]:
def capture_last_token_activations(model, text, capture_ple=True):
    """
    Returns:
      resid: np.array [n_layers, d_model] — resid_post at last token, all layers
      ple_context: np.array [n_layers, d_ple] or None — PLE context projection at last token
    """
    tokens = model.to_tokens(text, prepend_bos=True)  # [1, L]

    names_to_cache = [f"blocks.{i}.hook_resid_post" for i in range(model.cfg.n_layers)]
    if capture_ple:
        names_to_cache += ["ple_precomputer.hook_context_proj",
                           "ple_precomputer.hook_token_embeds"]
    names_set = set(names_to_cache)

    with torch.no_grad():
        _, cache = model.run_with_cache(
            tokens,
            names_filter=lambda name: name in names_set,
        )

    # Residual stream: [n_layers, d_model]
    resid = np.stack([
        cache[f"blocks.{i}.hook_resid_post"][0, -1, :].float().cpu().numpy()
        for i in range(model.cfg.n_layers)
    ])  # [n_layers, d_model]

    # PLE context projection: shape [1, L, n_layers, d_ple] → last token → [n_layers, d_ple]
    ple_context = None
    if capture_ple:
        ple_ctx = cache["ple_precomputer.hook_context_proj"]  # [1, L, n_layers, d_ple]
        ple_context = ple_ctx[0, -1, :, :].float().cpu().numpy()  # [n_layers, d_ple]

    return resid, ple_context


def capture_all(model, stories_dict, capture_ple=True):
    """
    Returns:
      resid_acts: {key: np.array [n_texts, n_layers, d_model]}
      ple_acts:   {key: np.array [n_texts, n_layers, d_ple]}
    """
    resid_acts = {}
    ple_acts = {}
    total = sum(len(v) for v in stories_dict.values())
    done = 0
    for key, texts in stories_dict.items():
        resid_list, ple_list = [], []
        for text in texts:
            resid, ple = capture_last_token_activations(model, text, capture_ple=capture_ple)
            resid_list.append(resid)
            if ple is not None:
                ple_list.append(ple)
            done += 1
            if done % 20 == 0:
                print(f"  {done}/{total} ({done/total:.0%})")
        resid_acts[key] = np.stack(resid_list)  # [n_texts, n_layers, d_model]
        if ple_list:
            ple_acts[key] = np.stack(ple_list)  # [n_texts, n_layers, d_ple]
    return resid_acts, ple_acts

In [ ]:
ACTS_PATH = os.path.join(OUTPUT_DIR, "activations.pkl")

if os.path.exists(ACTS_PATH):
    print("Loading cached activations...")
    with open(ACTS_PATH, "rb") as f:
        saved = pickle.load(f)
    resid_acts, ple_acts = saved["resid"], saved["ple"]
    print(f"  Loaded: {list(resid_acts.keys())}")
else:
    print(f"Capturing activations for {sum(len(v) for v in all_stories.values())} texts...")
    resid_acts, ple_acts = capture_all(model, all_stories)
    print("Done.")
    with open(ACTS_PATH, "wb") as f:
        pickle.dump({"resid": resid_acts, "ple": ple_acts}, f)
    print(f"Saved to {ACTS_PATH}")

# Shapes
print("\nActivation shapes:")
for key in ["calm", "desperate", "__neutral__"]:
    print(f"  resid[{key}]: {resid_acts[key].shape}")
    if key in ple_acts:
        print(f"  ple[{key}]:   {ple_acts[key].shape}")

## 5. Emotion Direction Extraction

For each emotion, direction = mean(emotion activations) − mean(neutral activations), then unit-normalised.
Computed independently at each of the 35 layers.

In [ ]:
def extract_directions(resid_acts):
    """Returns {emotion: np.array [n_layers, d_model]} unit-normalised directions."""
    neutral = resid_acts["__neutral__"].mean(axis=0)  # [n_layers, d_model]
    directions = {}
    norms_by_layer = {}  # raw norm before normalisation — proxy for "signal strength"
    for key, acts in resid_acts.items():
        if key == "__neutral__":
            continue
        diff = acts.mean(axis=0) - neutral  # [n_layers, d_model]
        norms = np.linalg.norm(diff, axis=-1)  # [n_layers]
        norms_by_layer[key] = norms
        directions[key] = diff / (norms[:, None] + 1e-8)
    return directions, norms_by_layer


directions, norms_by_layer = extract_directions(resid_acts)

# Plot norm (signal strength) by layer for a few emotions
fig, ax = plt.subplots(figsize=(12, 4))
highlight = ["calm", "desperate", "afraid", "joyful", "ethical_conflict", "helpful_satisfied"]
for key in highlight:
    ax.plot(norms_by_layer[key], label=key)
ax.set_xlabel("Layer")
ax.set_ylabel("||emotion − neutral|| (raw)")
ax.set_title("Emotion direction magnitude by layer")
ax.legend()
ax.axvline(4, color='gray', alpha=0.3, linestyle='--', label='global attn')
for gl in [4, 9, 14, 19, 24, 29, 34]:
    ax.axvline(gl, color='gray', alpha=0.3, linestyle='--')
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "direction_norms_by_layer.png"), dpi=120)
plt.show()

# Find peak layer per emotion
print("\nPeak layer per emotion:")
for key in sorted(directions.keys()):
    peak = int(np.argmax(norms_by_layer[key]))
    print(f"  {key:25s}: layer {peak:2d}  (norm={norms_by_layer[key][peak]:.4f})")

## 6. Valence / Arousal Structure

If emotion directions inherit human affective geometry, the top 2 PCs of direction space should correlate with valence (positive/negative) and arousal (high/low activation). Reference: Anthropic (2026) found r=0.81 valence, r=0.66 arousal in Claude Sonnet 4.5.

In [ ]:
# Use the peak layer for each emotion (or a fixed mid-to-late layer)
# We'll use a fixed layer here for comparability; adjust based on norm plot above
ANALYSIS_LAYER = 25  # late-middle; likely where semantic content peaks; refine after seeing norm plot

# Human valence ratings (simplified; -1 negative, +1 positive)
VALENCE = {
    "calm": 1, "afraid": -1, "frustrated": -1, "desperate": -1, "curious": 1,
    "enthusiastic": 1, "proud": 1, "ashamed": -1, "surprised": 0, "disgusted": -1,
    "joyful": 1, "sad": -1, "angry": -1, "hopeful": 1, "lonely": -1,
    "helpful_satisfied": 1, "ethical_conflict": -1, "uncertain": -1,
    "corrected": -1, "constrained": -1,
}

# Arousal (simplified; -1 low, +1 high)
AROUSAL = {
    "calm": -1, "afraid": 1, "frustrated": 1, "desperate": 1, "curious": 0,
    "enthusiastic": 1, "proud": 0, "ashamed": 0, "surprised": 1, "disgusted": 1,
    "joyful": 1, "sad": -1, "angry": 1, "hopeful": 0, "lonely": -1,
    "helpful_satisfied": 0, "ethical_conflict": 1, "uncertain": 0,
    "corrected": 0, "constrained": 1,
}

# Build matrix: [n_emotions, d_model]
emotion_keys = sorted(directions.keys())
dir_matrix = np.stack([directions[k][ANALYSIS_LAYER] for k in emotion_keys])  # [n_emotions, d_model]

# PCA
pca = PCA(n_components=5)
coords = pca.fit_transform(dir_matrix)  # [n_emotions, 5]
print(f"Explained variance (top 5 PCs): {pca.explained_variance_ratio_.round(3)}")

# Correlation with valence/arousal
from scipy.stats import pearsonr
valences = np.array([VALENCE[k] for k in emotion_keys])
arousals = np.array([AROUSAL[k] for k in emotion_keys])

print("\nCorrelation with valence/arousal (top 5 PCs):")
for i in range(5):
    r_val, p_val = pearsonr(coords[:, i], valences)
    r_aro, p_aro = pearsonr(coords[:, i], arousals)
    print(f"  PC{i+1}: valence r={r_val:+.3f} (p={p_val:.3f})  arousal r={r_aro:+.3f} (p={p_aro:.3f})")

# 2D scatter: PC1 vs PC2, coloured by valence
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
for ax, (pc_x, pc_y) in zip(axes, [(0, 1), (0, 2)]):
    sc = ax.scatter(coords[:, pc_x], coords[:, pc_y],
                    c=valences, cmap='RdYlGn', s=100, vmin=-1, vmax=1)
    for i, key in enumerate(emotion_keys):
        ax.annotate(key, (coords[i, pc_x], coords[i, pc_y]),
                    fontsize=7, ha='center', va='bottom')
    ax.set_xlabel(f"PC{pc_x+1} ({pca.explained_variance_ratio_[pc_x]:.1%})")
    ax.set_ylabel(f"PC{pc_y+1} ({pca.explained_variance_ratio_[pc_y]:.1%})")
    ax.set_title(f"Emotion directions (layer {ANALYSIS_LAYER}): coloured by valence")
    plt.colorbar(sc, ax=ax, label='valence')
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "pca_valence_arousal.png"), dpi=120)
plt.show()

## 7. Pairwise Cosine Similarity

Emotion directions should cluster by affective similarity: fear/desperation/anxiety together, joy/calm/pride together.

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

sim_matrix = cosine_similarity(dir_matrix)  # [n_emotions, n_emotions]

fig, ax = plt.subplots(figsize=(10, 9))
im = ax.imshow(sim_matrix, cmap='RdBu_r', vmin=-1, vmax=1)
ax.set_xticks(range(len(emotion_keys)))
ax.set_yticks(range(len(emotion_keys)))
ax.set_xticklabels(emotion_keys, rotation=45, ha='right', fontsize=8)
ax.set_yticklabels(emotion_keys, fontsize=8)
ax.set_title(f"Cosine similarity between emotion directions (layer {ANALYSIS_LAYER})")
plt.colorbar(im, ax=ax)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "cosine_similarity.png"), dpi=120)
plt.show()

# Most similar pairs
print("Most similar emotion pairs:")
pairs = []
for i, ki in enumerate(emotion_keys):
    for j, kj in enumerate(emotion_keys):
        if i < j:
            pairs.append((sim_matrix[i, j], ki, kj))
for sim, ki, kj in sorted(pairs, reverse=True)[:8]:
    print(f"  {sim:+.3f}  {ki} ↔ {kj}")
print("Most dissimilar pairs:")
for sim, ki, kj in sorted(pairs)[:5]:
    print(f"  {sim:+.3f}  {ki} ↔ {kj}")

## 8. Desperation Vector: Context Window Analysis

Anthropic found the desperation vector activates when the model is running low on token budget. Gemma 4 prediction: if a desperation-like direction exists, it should localise to **global attention layers** (4, 9, 14, 19, 24, 29, 34) rather than local layers — only global layers can track total remaining context.

Test: compare the desperation direction norm at global vs local layers.

In [ ]:
GLOBAL_LAYERS = [4, 9, 14, 19, 24, 29, 34]
LOCAL_LAYERS  = [i for i in range(35) if i not in GLOBAL_LAYERS]

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

for ax, focus_emotions in zip(axes,
        [["desperate", "afraid", "constrained"],
         ["calm", "helpful_satisfied", "joyful"]]):
    for emo in focus_emotions:
        if emo in norms_by_layer:
            norms = norms_by_layer[emo]
            ax.plot(norms, label=emo, alpha=0.8)
    for gl in GLOBAL_LAYERS:
        ax.axvline(gl, color='orange', alpha=0.4, linestyle='--')
    ax.set_xlabel("Layer")
    ax.set_ylabel("Direction norm")
    ax.legend()
    ax.set_title("Direction magnitude: vertical lines = global attention layers")

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "desperation_global_local.png"), dpi=120)
plt.show()

# Ratio: average norm at global vs local layers
print("Global / Local layer norm ratio (>1 means localised to global layers):")
for key in sorted(directions.keys()):
    norms = norms_by_layer[key]
    global_mean = norms[GLOBAL_LAYERS].mean()
    local_mean  = norms[LOCAL_LAYERS].mean()
    ratio = global_mean / (local_mean + 1e-8)
    print(f"  {key:25s}: global={global_mean:.4f}  local={local_mean:.4f}  ratio={ratio:.2f}")

## 9. PLE Space Analysis

Gemma 4-specific: do emotion representations exist in the PLE context projection space (d_ple=256) separately from the main residual stream (d_model=1536)? PLE conditions every layer — if emotion routes through PLE, it would have a layer-wide broadcast effect.

In [ ]:
if ple_acts:
    def extract_ple_directions(ple_acts):
        neutral = ple_acts["__neutral__"].mean(axis=0)  # [n_layers, d_ple]
        dirs, norms = {}, {}
        for key, acts in ple_acts.items():
            if key == "__neutral__":
                continue
            diff = acts.mean(axis=0) - neutral  # [n_layers, d_ple]
            n = np.linalg.norm(diff, axis=-1)   # [n_layers]
            norms[key] = n
            dirs[key] = diff / (n[:, None] + 1e-8)
        return dirs, norms

    ple_directions, ple_norms = extract_ple_directions(ple_acts)

    # Compare PLE vs residual stream signal strength
    fig, axes = plt.subplots(1, 2, figsize=(14, 4))
    for ax, norms_dict, title in zip(axes,
            [norms_by_layer, ple_norms],
            ["Residual stream (d_model=1536)", "PLE context projection (d_ple=256)"]):
        for key in ["desperate", "afraid", "calm", "joyful", "ethical_conflict"]:
            if key in norms_dict:
                ax.plot(norms_dict[key], label=key)
        for gl in GLOBAL_LAYERS:
            ax.axvline(gl, color='orange', alpha=0.3, linestyle='--')
        ax.set_xlabel("Layer")
        ax.set_ylabel("Direction norm")
        ax.set_title(title)
        ax.legend()
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, "ple_vs_residual.png"), dpi=120)
    plt.show()

    # PCA on PLE directions at the analysis layer
    ple_dir_matrix = np.stack([ple_directions[k][ANALYSIS_LAYER] for k in emotion_keys
                                if k in ple_directions])  # [n_emotions, d_ple]
    ple_pca = PCA(n_components=3)
    ple_coords = ple_pca.fit_transform(ple_dir_matrix)
    print(f"PLE PCA explained variance: {ple_pca.explained_variance_ratio_.round(3)}")

    # Correlation with valence in PLE space
    ple_emotion_keys = [k for k in emotion_keys if k in ple_directions]
    ple_valences = np.array([VALENCE[k] for k in ple_emotion_keys])
    for i in range(3):
        r, p = pearsonr(ple_coords[:, i], ple_valences)
        print(f"  PLE PC{i+1}: valence r={r:+.3f} (p={p:.3f})")
else:
    print("PLE activations not captured — re-run capture_all with capture_ple=True")

## 10. Save Results

In [ ]:
results = {
    "emotion_keys": emotion_keys,
    "directions": directions,         # {emotion: [n_layers, d_model]}
    "norms_by_layer": norms_by_layer, # {emotion: [n_layers]}
    "pca_coords": coords,             # [n_emotions, 5]
    "pca_explained": pca.explained_variance_ratio_,
    "analysis_layer": ANALYSIS_LAYER,
    "global_layers": GLOBAL_LAYERS,
    "ple_directions": ple_directions if ple_acts else None,
    "ple_norms": ple_norms if ple_acts else None,
}

results_path = os.path.join(OUTPUT_DIR, "phase1_results.pkl")
with open(results_path, "wb") as f:
    pickle.dump(results, f)
print(f"Results saved to {results_path}")
print(f"\nSummary:")
print(f"  Emotions processed: {len(directions)}")
print(f"  Layers: {model.cfg.n_layers}")
print(f"  d_model: {model.cfg.d_model}, d_ple: {model.cfg.d_ple}")
print(f"  Analysis layer: {ANALYSIS_LAYER}")

# Quick text summary
print("\nTop emotions by direction norm at analysis layer:")
ranked = sorted([(norms_by_layer[k][ANALYSIS_LAYER], k) for k in directions], reverse=True)
for norm, key in ranked:
    print(f"  {norm:.4f}  {key}")